# ClimaX: A Foundation Model for Weather and Climate
## Interactive Research Review Notebook

Companion notebook for Nguyen et al. (ICML 2023), *ClimaX: A foundation model for weather and climate*.

**Purpose:** Walk through the paper’s problem framing, architecture ideas, and evaluation logic using transparent **synthetic** benchmarks. This notebook does **not** download CMIP6/ERA5 or load official `microsoft/ClimaX` checkpoints.

**Paper:** https://arxiv.org/abs/2301.10343  
**Code (official):** https://github.com/microsoft/ClimaX

## 1. Learning objectives

1. Explain why foundation models matter for heterogeneous weather/climate data.
2. Relate **variable tokenization** and **variable aggregation** to Vision Transformers.
3. Reproduce an educational skill/evaluation pipeline (forecast → projection → downscaling → scaling).
4. Apply descriptive statistics and paired *t*-tests to compare synthetic model errors.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Reuse the project analysis module for a single source of truth
import climax_foundation_analysis as cfa

sns.set_theme(style="whitegrid", context="notebook")
ROOT = Path(".").resolve()
print("Project root:", ROOT)

## 2. Paper snapshot

| Aspect | ClimaX design |
|---|---|
| Backbone | Vision Transformer (ViT) |
| Key blocks | Variable tokenization + variable aggregation |
| Pretraining data | Heterogeneous CMIP6-derived climate simulations |
| Objective | Self-supervised / forecasting-style pretraining |
| Finetuning tasks | Global/regional forecast, ClimateBench projection, downscaling |
| Claim | Generality from heterogeneous pretraining improves downstream skill |

ClimaX treats atmospheric variables as modalities so irregular datasets can share one architecture—analogous to multimodal fusion in scientific ML and omics.

## 3. Build synthetic forecast-error dataset

We simulate absolute errors for three model families across lead times and variables:

- `nwp_like` — strong short range
- `task_specific_dl` — strong short/medium range, faster long-lead degradation
- `climax_like` — competitive short range, better long-horizon transfer (paper narrative)

In [ ]:
cfg = cfa.Config()
forecast_df = cfa.build_forecast_dataset(cfg)
print(forecast_df.head())
print("\nShape:", forecast_df.shape)
print("Models:", sorted(forecast_df["model"].unique()))
print("Variables:", sorted(forecast_df["variable"].unique()))
print("Lead hours:", sorted(forecast_df["lead_hours"].unique()))

## 4. Descriptive statistics

Report **mean** (MAE), **median**, **standard deviation**, and RMSE-style summaries by model × variable × lead time.

In [ ]:
summary = cfa.summarize_forecast_skill(forecast_df)
display_cols = ["model", "variable", "lead_hours", "mae", "median_ae", "std_ae", "rmse"]
summary.loc[summary["variable"] == "t2m", display_cols].head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(
    data=summary[summary["variable"] == "t2m"],
    x="lead_hours",
    y="mae",
    hue="model",
    marker="o",
    ax=ax,
)
ax.set_title("Synthetic T2m forecast skill (lower MAE is better)")
ax.set_xlabel("Lead time (hours)")
ax.set_ylabel("MAE (a.u.)")
plt.tight_layout()
plt.show()

## 5. Inferential statistics: paired *t*-tests

For each variable and lead time, test whether `task_specific_dl` absolute errors are **greater** than `climax_like` errors (paired samples).

- $H_0$: mean difference = 0
- $H_1$: mean(`task_specific_dl` − `climax_like`) > 0
- Report *t* statistic and *p*-value

In [ ]:
ttests = cfa.paired_ttests_climax_vs_baseline(forecast_df)
t2m_tests = ttests[ttests["variable"] == "t2m"].copy()
t2m_tests["significant_0.05"] = t2m_tests["p_value"] < 0.05
t2m_tests

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
colors = np.where(t2m_tests["significant_0.05"], "#2a9d8f", "#adb5bd")
ax.bar(t2m_tests["lead_hours"].astype(str), -np.log10(t2m_tests["p_value"]), color=colors)
ax.axhline(-np.log10(0.05), color="crimson", linestyle="--", label="p = 0.05")
ax.set_title("T2m paired test strength (−log10 p); teal = significant")
ax.set_xlabel("Lead time (hours)")
ax.set_ylabel("-log10(p-value)")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Climate projection transfer (ClimateBench-style)

The paper emphasizes that ClimaX attention layers transfer to ClimateBench even when input/output variables were unseen in pretraining. Below: synthetic scores (higher is better).

In [ ]:
climatebench = cfa.build_climatebench_scores(cfg)
climatebench.pivot(index="model", columns="target", values="score")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=climatebench, x="target", y="score", hue="model", ax=ax)
ax.set_ylim(0, 1.05)
ax.set_title("Synthetic ClimateBench-style scores (higher is better)")
plt.tight_layout()
plt.show()

## 7. Climate downscaling comparison

Downscaling maps coarse climate-model fields to finer reanalysis-like targets. Synthetic RMSE (lower is better).

In [ ]:
downscaling = cfa.build_downscaling_metrics(cfg)
display(downscaling)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=downscaling, x="variable", y="rmse", hue="model", ax=ax)
ax.set_title("Synthetic downscaling RMSE (lower is better)")
plt.tight_layout()
plt.show()

## 8. Scaling illustration

Foundation-model literature often reports predictable gains from more parameters and data. Toy curve for 3-day T850 MAE vs model size.

In [ ]:
scaling = cfa.build_scaling_curve(cfg)
display(scaling)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(scaling["params_millions"], scaling["mae_3day_t850"], marker="o", linewidth=2)
ax.set_xscale("log")
ax.set_xlabel("Parameters (millions, log scale)")
ax.set_ylabel("3-day T850 MAE (a.u.)")
ax.set_title("Synthetic scaling law illustration")
plt.tight_layout()
plt.show()

## 9. Export artifacts for the blog post / README

Run the full pipeline (same as `python climax_foundation_analysis.py`).

In [ ]:
exit_code = cfa.main()
print("Pipeline exit code:", exit_code)
print("Check outputs/ and data/processed/ for CSV + PNG artifacts.")

## 10. Discussion prompts (for coursework)

1. How does variable aggregation control Transformer compute as variable count grows?
2. Why might CMIP6 heterogeneity help more than training only on ERA5 for a single task?
3. What ethical risks arise if foundation weather models are deployed without calibrated uncertainty?
4. How would you replace synthetic errors with real ClimaX vs IFS forecast fields?

### Upgrade path to real data

1. Download WeatherBench/ERA5 and ClimateBench following official ClimaX docs.
2. Load pretrained checkpoints from the authors’ release.
3. Keep the same metric and plotting interfaces; swap only data loaders.
4. Report confidence intervals and extreme-event metrics in addition to MAE/RMSE.

## References

- Nguyen, T., Brandstetter, J., Kapoor, A., Gupta, J. K., & Grover, A. (2023). ClimaX: A foundation model for weather and climate. *ICML*. arXiv:2301.10343
- Microsoft Research ClimaX announcement and documentation
- Watson-Parris et al. (2022). ClimateBench v1.0
- Hersbach et al. ERA5 reanalysis
- Eyring et al. (2016). CMIP6 overview